# Heart Disease Risk Predictor
Hi, My name is Joaquin Carmona, and I'll be working on this project to predict the risk of heart disease. This notebook will be used to explore the data and perform EDA. I'll be learning as I go and using this notebook to document my progress, videos or documents I find useful, and any other resources I find helpful.  

My purpose with this project is to jump in the Healthcare AI world doing this basic project to get started. I'll be using different approaches and algorithms to predict the risk of heart disease.

## Dataset
This is a multivariate type of dataset which means providing or involving a variety of separate mathematical or statistical variables, multivariate numerical data analysis. It is composed of 14 attributes which are age, sex, chest pain type, resting blood pressure, serum cholesterol, fasting blood sugar, resting electrocardiographic results, maximum heart rate achieved, exercise-induced angina, oldpeak — ST depression induced by exercise relative to rest, the slope of the peak exercise ST segment, number of major vessels and Thalassemia. This database includes 76 attributes, but all published studies relate to the use of a subset of 14 of them. The Cleveland database is the only one used by ML researchers to date. One of the major tasks on this dataset is to predict based on the given attributes of a patient that whether that particular person has heart disease or not and other is the experimental task to diagnose and find out various insights from this dataset which could help in understanding the problem more.

## Reference
Detrano, R. et al. (1989). *International application of a new probability algorithm for the 
diagnosis of coronary artery disease.* American Journal of Cardiology, 64(5), 304-310.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('heart_disease_uci.csv')

In [3]:
print(df.isnull().sum())
#first we check for null values, as we can see there are a lot of null values in the dataset, we will need to handle them, and 
# take the best decision for each case.

print(f"Unique values per column:\n{df.nunique()}")



id            0
age           0
sex           0
dataset       0
cp            0
trestbps     59
chol         30
fbs          90
restecg       2
thalch       55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
num           0
dtype: int64
Unique values per column:
id          920
age          50
sex           2
dataset       4
cp            4
trestbps     61
chol        217
fbs           2
restecg       3
thalch      119
exang         2
oldpeak      53
slope         3
ca            4
thal          3
num           5
dtype: int64


In [4]:
print(df.columns)

Index(['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs',
       'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'],
      dtype='str')


## Dataset Variables 

The dataset contains 920 patients from 4 different institutions and 16 variables:

| Variable | Type | Clinical Description |
|----------|------|-------------------|
| `id` | int | Unique patient identifier. No predictive value. |
| `age` | int | Age in years. Well-established cardiovascular risk factor. |
| `sex` | str | Biological sex: Male / Female. Men face higher risk at younger ages. |
| `dataset` | str | Source institution: Cleveland, Hungary, Switzerland, VA Long Beach. |
| `cp` | str | Chest pain type: typical angina, atypical angina, non-anginal, asymptomatic. Asymptomatic pain is paradoxically the most associated with severe disease. |
| `trestbps` | float | Resting blood pressure at hospital admission (mm Hg). Hypertension is a major risk factor. |
| `chol` | float | Serum cholesterol (mg/dl). High levels associated with arterial plaque buildup. |
| `fbs` | object | Fasting blood sugar > 120 mg/dl: True/False. Indicates diabetes, a critical comorbidity. |
| `restecg` | str | Resting electrocardiogram results: normal, ST-T wave abnormality, left ventricular hypertrophy. |
| `thalch` | float | Maximum heart rate achieved during stress test. Lower capacity = higher risk. |
| `exang` | object | Exercise-induced angina: True/False. Pain triggered by exertion signals ischemia. |
| `oldpeak` | float | ST segment depression induced by exercise relative to rest. Indicates myocardial ischemia. |
| `slope` | str | Slope of the peak exercise ST segment: upsloping, flat, downsloping. Downsloping is the most concerning. |
| `ca` | float | Number of major vessels colored by fluoroscopy (0-3). More obstructed vessels = greater severity. |
| `thal` | str | Thallium stress test result (blood flow to heart): normal, fixed defect, reversible defect. |
| `num` | int | **Target variable.** Heart disease diagnosis: 0 = absent, 1-4 = present (severity). |


Age is the most important risk factor in developing cardiovascular or heart diseases, with approximately a tripling of risk with each decade of life. Coronary fatty streaks can begin to form in adolescence. It is estimated that 82 percent of people who die of coronary heart disease are 65 and older. Simultaneously, the risk of stroke doubles every decade after age 55.

Men are at greater risk of heart disease than pre-menopausal women. Once past menopause, it has been argued that a woman’s risk is similar to a man’s although more recent data from the WHO and UN disputes this. If a female has diabetes, she is more likely to develop heart disease than a male with diabetes.

In [5]:
df.head()
#TODO SEPARATE BAR CHART 1 = HEART DISEASE 0= IT IS NOT HEART DISEASE
#SEPARATE FROM MAN AND WOMAN WITH HEART DISEASE

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


Resources I used:



https://www.kaggle.com/code/muhammadhanzla1234/heart-disease-prediction-with-77-55-accuracy#Context

https://www.kaggle.com/code/hussain99/heart-disease-prediction-10-models-eda